In [85]:
import os
import cv2
import numpy as np

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report


# =========================
# 1. قراءة الصور من المجلدات
# =========================

def load_images_from_folder(folder_path, label, image_size=(128, 128)):
    images = []
    labels = []

    for filename in os.listdir(folder_path):
        path = os.path.join(folder_path, filename)

        img = cv2.imread(path)

        if img is None:
            continue

        # تحويل إلى رمادي
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # توحيد الحجم
        img = cv2.resize(img, image_size)

        images.append(img)
        labels.append(label)

    return images, labels


cats_images, cats_labels = load_images_from_folder("dogs-vs-cats/cats", 0)
dogs_images, dogs_labels = load_images_from_folder("dogs-vs-cats/Dogs", 1)


In [91]:
dogs_images[0].shape
cats_labels[0]


0

In [75]:
len(cats_images)

640

In [76]:
len(dogs_images)

753

In [77]:

images = cats_images + dogs_images
labels = cats_labels + dogs_labels

images = np.array(images)
labels = np.array(labels)

print("Number of images:", len(images))


# =========================
# 2. استخراج الميزات HOG
# =========================

features = []

for img in images:
    feature = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    features.append(feature)



Number of images: 1393


In [93]:
len(features)

1393

In [95]:
features[0].shape

(8100,)

In [79]:

X = np.array(features)
y = labels

print("Features shape:", X.shape)


Features shape: (1393, 8100)


In [97]:

# =========================
# 3. تقسيم البيانات
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51,
    stratify=y
)


# =========================
# 4. بناء وتدريب الخوارزمية
# =========================

model = SVC(kernel="linear")

model.fit(X_train, y_train)


# =========================
# 5. التنبؤ وقياس الدقة
# =========================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Accuracy: 0.6379928315412187

Classification Report:
              precision    recall  f1-score   support

         Cat       0.62      0.54      0.58       128
         Dog       0.65      0.72      0.68       151

    accuracy                           0.64       279
   macro avg       0.64      0.63      0.63       279
weighted avg       0.64      0.64      0.63       279



In [81]:
test=(y_pred,y_test)
test

(array([0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
        1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0,
        0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1,
        0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0,
        0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0,
        0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1,
        1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0,
        1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1]),
 array([0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0

In [99]:


# =========================
# 6. تجربة التنبؤ على صورة جديدة
# =========================

def predict_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        print("Image not found")
        return

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (128, 128))

    feature = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    prediction = model.predict([feature])[0]

    if prediction == 0:
        print("Prediction: Cat")
    else:
        print("Prediction: Dog")


# مثال:
predict_image("dogs-vs-cats/dogs/dog.120.jpg")

Prediction: Dog


In [101]:
predict_image("dogs-vs-cats/cats/cat.12200.jpg")

Prediction: Cat
